# 06 · Rolling-origin backtest

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

One train/test split is one roll of the dice. A **rolling-origin backtest** repeats it: train on everything before a cutoff, forecast the next three months, move the cutoff forward, repeat. It is how you would show a pricing or retention committee that a model holds up over time.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import sys
sys.path.insert(0, "../python")

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulate as sim
importlib.reload(sim)

members = pd.read_parquet("../data/members.parquet")
print(f"{len(members):,} members, {members['lapsed'].mean():.1%} lapsed")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATS = list(sim.LEVELS)
CAT_LEVELS = [sim.LEVELS[c][0] for c in CATS]
NUMS = ["premium_increase", "tenure", "early", "anniversary", "long"]
FEATURE_COLS = CATS + NUMS
CUTOFF = "2025-01"   # train on months before this, test on this month onwards


def make_prep(dense=False):
    """One-hot the categories (reference level dropped) and scale the numbers."""
    return ColumnTransformer([
        ("cat", OneHotEncoder(categories=CAT_LEVELS, drop="first", sparse_output=not dense), CATS),
        ("num", StandardScaler(), NUMS),
    ])

pm = sim.person_months(members)
CUTOFFS = ["2024-07", "2024-10", "2025-01", "2025-04"]
HORIZON = 3   # months forecast after each cutoff

## 1. The backtest loop

### Your turn 1.1

For each cutoff:

1. `train` = rows with `calendar_month < cutoff`
2. `window` = the next `HORIZON` months: `cutoff` up to, but not including, `str(pd.Period(cutoff, "M") + HORIZON)`
3. Fit a logistic pipeline (`make_prep()` and `LogisticRegression(max_iter=1000)`) on `train`, and predict for `window`
4. Record the cutoff, the AUC, and A/E (actual lapses ÷ sum of predictions)

Collect one dict per fold in a list, and turn it into `backtest`, a DataFrame. Also keep every fold's window rows with their predictions in `fold_preds`, a list of DataFrames, for section 2.

In [ ]:
rows = []
fold_preds = []
for cutoff in CUTOFFS:
    ...
backtest = ...

In [ ]:
check("1.1 one row per cutoff", lambda: len(backtest), len(CUTOFFS))
check("1.1 AUC stable across folds", lambda: backtest["auc"].max() - backtest["auc"].min() < 0.05)
check("1.1 every fold's A/E within 15%", lambda: backtest["ae"].between(0.85, 1.15).all())

## 2. Actual vs expected by month

The chart a retention committee would actually look at: expected and actual lapses for each forecast month, across all folds.

### Your turn 2.1

Concatenate `fold_preds`, group by `calendar_month`, and build `monthly` with columns `actual` (sum of `lapse`) and `expected` (sum of `pred`). Plot both as lines, then add `ae` = actual ÷ expected.

In [ ]:
monthly = ...

In [ ]:
check("2.1 twelve forecast months", lambda: len(monthly), len(CUTOFFS) * HORIZON)
check("2.1 monthly A/E mostly within 20%", lambda: monthly["ae"].between(0.8, 1.2).mean() >= 0.75)

Look for months where actual jumps above expected: anniversary-heavy months will stand out if the model under-weights renewals. On real data, a run of A/E above 1 is the early warning that lapse experience is worsening.

## Wrap-up

Commit from VS Code: `Rolling-origin backtest with monthly A/E`.